# 🎓 AI Code Mentor — Flask API Server (Kaggle Edition)

This notebook runs the Flask API backend for the AI Code Mentor project entirely on Kaggle's free GPU.

**Before you start:**
- Turn on **GPU T4 x2** from the right-side panel Settings.
- Turn on **Internet** from the right-side panel Settings.

**Steps:**
1. Install dependencies
2. Authenticate with HuggingFace
3. Upload project files to `/kaggle/working/`
4. Start Flask API + ngrok tunnel

In [ ]:
# Step 1: Install all dependencies
!pip install -q -U bitsandbytes accelerate transformers pylint \
    langchain langchain-community langchain-core \
    langchain-text-splitters langchain-huggingface \
    faiss-cpu rank_bm25 beautifulsoup4 huggingface_hub \
    sentence-transformers flask flask-cors pyngrok

In [ ]:
# Step 2: Authenticate with HuggingFace
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print('[OK] Got token from Kaggle Secrets')
except Exception:
    hf_token = os.getenv('HF_TOKEN', '')
    if not hf_token:
        hf_token = input('Enter your HuggingFace token: ')

if hf_token:
    login(token=hf_token)
    os.environ['HF_TOKEN'] = hf_token
    print('[OK] Logged in to HuggingFace')
else:
    print('[WARNING] No token provided')

## Step 3: Upload Project Files

To upload files on Kaggle, look at the **Right Panel**.
Navigate to **Data > Output > /kaggle/working**. 
Click the **Upload (arrow)** button and upload these exactly:
- `code_reviewer.py`
- `api.py`
- `knowledge_base.md`
- `database.py`

In [ ]:
import os
import shutil

# Search for our files anywhere in the Input folder and copy them to Working
needed = ['code_reviewer.py', 'api.py', 'knowledge_base.md', 'database.py']

for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        if file in needed:
            shutil.copy(os.path.join(root, file), '/kaggle/working/')
            print(f"✅ Auto-copied {file} to working directory")


In [ ]:
# Check if files are uploaded successfully to /kaggle/working
import os

needed = ['code_reviewer.py', 'api.py', 'knowledge_base.md', 'database.py']
missing = [f for f in needed if not os.path.exists(f'/kaggle/working/{f}')]

if missing:
    print(f'Please manually upload the following files using the right panel: {missing}')
else:
    print('[OK] All core files are securely present in /kaggle/working!')

for f in needed:
    status = '✅' if os.path.exists(f'/kaggle/working/{f}') else '❌'
    print(f'{status} {f}')


In [ ]:
!pip install pyngrok

In [ ]:
# Step 4: Configure Cloudflare for tunneling (Ngrok Alternative)
import os
import time

# 1. Download Cloudflare's tunneling tool (Hosted on GitHub, no 403 errors!)
if not os.path.exists('cloudflared'):
    print("⏳ Downloading Cloudflare Tunnel...")
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
    os.system('chmod +x cloudflared')
    print("✅ Cloudflare downloaded successfully!")

# 2. Start the tunnel in the background
# (Change 5000 to whatever port your Flask/FastAPI app runs on)
PORT = 5000
print(f"🚀 Starting tunnel on port {PORT}...")

# Cloudflare writes its URL to standard error, so we pipe it to a log file
os.system(f'nohup ./cloudflared tunnel --url http://localhost:{PORT} > cloudflare_tunnel.log 2>&1 &')

# Wait a few seconds for the tunnel to establish connection
time.sleep(5)

# 3. Extract and print the public URL
try:
    with open('cloudflare_tunnel.log', 'r') as f:
        logs = f.read()
        
    # Search the logs for the generated 'trycloudflare.com' URL
    url = None
    for line in logs.split('\n'):
        if 'trycloudflare.com' in line:
            # Extract just the URL string
            url_start = line.find('https://')
            if url_start != -1:
                url = line[url_start:].split()[0]
                break
                
    if url:
        print(f"\n🎉 Tunnel established!")
        print(f"👉 Your Public API URL is: {url}")
    else:
        print("\n[WARNING] Could not find the URL in the logs. The tunnel might still be starting.")
        print("Run this cell again in a few seconds, or check 'cloudflare_tunnel.log'.")
        
except Exception as e:
    print(f"Error reading logs: {e}")

In [ ]:
# Step 5: Start Flask API Server
import subprocess
import threading
import urllib.request
import time
import os
import shutil

# ── 0. Auto-copy project files (resilient to session restarts) ─────────
needed = ['code_reviewer.py', 'api.py', 'knowledge_base.md', 'database.py']
for root, dirs, files in os.walk('/kaggle/input/'):
    for file in files:
        if file in needed:
            dst = f'/kaggle/working/{file}'
            shutil.copy(os.path.join(root, file), dst)
            print(f'✅ Copied {file}')

missing = [f for f in needed if not os.path.exists(f'/kaggle/working/{f}')]
if missing:
    print(f'❌ Missing files (upload them via the Data panel): {missing}')
    raise FileNotFoundError(f'Missing: {missing}')

# ── 1. Kill stale processes ────────────────────────────────────────────
os.system('pkill -f api.py')
time.sleep(2)

# ── 2. Start api.py with live output piped to this cell ───────────────
print('🚀 Starting Flask API server...')
process = subprocess.Popen(
    ['python', '-u', '/kaggle/working/api.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream every line from api.py into the notebook cell
def _stream():
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)

threading.Thread(target=_stream, daemon=True).start()

# ── 3. Wait for server to be ready (health check) ─────────────────────
ready = False
for _ in range(96):        # up to 8 min
    if process.poll() is not None:
        print(f'\n❌ api.py exited (code {process.returncode}) — see output above.')
        break
    try:
        urllib.request.urlopen('http://localhost:5000/api/health', timeout=3)
        ready = True
        break
    except Exception:
        time.sleep(5)

# ── 4. Print public URL ────────────────────────────────────────────────
if ready:
    public_url = 'NOT FOUND — rerun Cell 8 (Cloudflare)'
    try:
        with open('cloudflare_tunnel.log') as f:
            for line in f:
                if 'trycloudflare.com' in line:
                    start = line.find('https://')
                    if start != -1:
                        public_url = line[start:].split()[0]
                        break
    except Exception:
        pass
    print(f'\n{"="*60}')
    print(f'✅ API is live at: {public_url}')
    print(f'{"="*60}')
else:
    print('\n❌ Server did not start. Check the streamed output above for the crash reason.')


In [ ]:
# (Optional) View server logs
import subprocess
result = subprocess.run(['tail', '-20', '/proc/' + str(process.pid) + '/fd/1'], 
                       capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
else:
    print('Server is running. Check the output of the previous cell for logs.')

In [ ]:
# Run this cell ONLY when you want to stop the server
import subprocess
import gc
import torch

subprocess.run(['pkill', '-f', 'api.py'], capture_output=True)
ngrok.kill()

gc.collect()
torch.cuda.empty_cache()
print('[OK] Server stopped and GPU memory cleared')